# Question Answering with LangChain, OpenAI, and MultiQuery Retriever

This interactive workbook demonstrates example of Elasticsearch's [MultiQuery Retriever](https://api.python.langchain.com/en/latest/retrievers/langchain.retrievers.multi_query.MultiQueryRetriever.html) to generate similar queries for a given user input and apply all queries to retrieve a larger set of relevant documents from a vectorstore.

Before we begin, we first split the fictional workplace documents into passages with `langchain` and uses OpenAI to transform these passages into embeddings and then store these into Elasticsearch.

We will then ask a question, generate similar questions using langchain and OpenAI, retrieve relevant passages from the vector store, and use langchain and OpenAI again to provide a summary for the questions.

## Install packages and import modules

In [1]:
!python3 -m pip install -qU jq lark langchain langchain-elasticsearch langchain_openai tiktoken

from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_elasticsearch import ElasticsearchStore
from langchain_openai.llms import OpenAI
from langchain.retrievers.multi_query import MultiQueryRetriever
from getpass import getpass

## Connect to Elasticsearch

ℹ️ We're using an Elastic Cloud deployment of Elasticsearch for this notebook. If you don't have an Elastic Cloud deployment, sign up [here](https://cloud.elastic.co/registration?utm_source=github&utm_content=elasticsearch-labs-notebook) for a free trial. 

We'll use the **Cloud ID** to identify our deployment, because we are using Elastic Cloud deployment. To find the Cloud ID for your deployment, go to https://cloud.elastic.co/deployments and select your deployment.

We will use [ElasticsearchStore](https://api.python.langchain.com/en/latest/vectorstores/langchain.vectorstores.elasticsearch.ElasticsearchStore.html) to connect to our elastic cloud deployment, This would help create and index data easily.  We would also send list of documents that we created in the previous step

In [4]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Get credentials from environment variables
ELASTIC_CLOUD_ID = os.getenv("ELASTIC_CLOUD_ID")
ELASTIC_API_KEY = os.getenv("ELASTIC_API_KEY") 
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

embeddings = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)

vectorstore = ElasticsearchStore(
    es_cloud_id=ELASTIC_CLOUD_ID,
    es_api_key=ELASTIC_API_KEY,
    index_name="workplace_docs_multiquery",  # give it a meaningful name
    embedding=embeddings,
)

## Indexing Data into Elasticsearch
Let's download the sample dataset and deserialize the document.

In [5]:
from urllib.request import urlopen
import json

url = "https://raw.githubusercontent.com/elastic/elasticsearch-labs/main/example-apps/chatbot-rag-app/data/data.json"

response = urlopen(url)
data = json.load(response)

with open("temp.json", "w") as json_file:
    json.dump(data, json_file)

### Split Documents into Passages

We’ll chunk documents into passages in order to improve the retrieval specificity and to ensure that we can provide multiple passages within the context window of the final question answering prompt.

Here we are chunking documents into 800 token passages with an overlap of 400 tokens.

Here we are using a simple splitter but Langchain offers more advanced splitters to reduce the chance of context being lost.

In [8]:
from langchain_community.document_loaders import JSONLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter


def metadata_func(record: dict, metadata: dict) -> dict:
    """
    Populate the metadata dictionary with keys name, summary, url, category, and updated_at.
    """
    # Extract and populate metadata fields from the record
    metadata["name"] = record.get("name", "")
    metadata["summary"] = record.get("summary", "")
    metadata["url"] = record.get("url", "")
    metadata["category"] = record.get("category", "")
    metadata["updated_at"] = record.get("updated_at", "")
    
    return metadata


# For more loaders https://python.langchain.com/docs/modules/data_connection/document_loaders/
# And 3rd party loaders https://python.langchain.com/docs/modules/data_connection/document_loaders/#third-party-loaders
loader = JSONLoader(
    file_path="temp.json",
    jq_schema=".[]",
    content_key="content",
    metadata_func=metadata_func,
)

# Define chunk size (800 tokens) and chunk overlap (400 tokens) as mentioned in the notebook
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=800, 
    chunk_overlap=400
)
docs = loader.load_and_split(text_splitter=text_splitter)

### Bulk Import Passages

Now that we have split each document into the chunk size of 800, we will now index data to elasticsearch using [ElasticsearchStore.from_documents](https://api.python.langchain.com/en/latest/vectorstores/langchain.vectorstores.elasticsearch.ElasticsearchStore.html#langchain.vectorstores.elasticsearch.ElasticsearchStore.from_documents).

We will use Cloud ID, Password and Index name values set in the `Create cloud deployment` step.

In [11]:
# Task 3: Alternative approach - create vectorstore first, then add documents
# Create the vectorstore instance first
vectorstore = ElasticsearchStore(
    es_cloud_id=ELASTIC_CLOUD_ID,
    es_api_key=ELASTIC_API_KEY,
    index_name="workplace_docs_multiquery_v2",  # use a new index name
    embedding=embeddings,
)

# Add documents in smaller batches to avoid bulk indexing issues
try:
    # Try adding documents in smaller chunks
    batch_size = 5
    for i in range(0, len(docs), batch_size):
        batch = docs[i:i + batch_size]
        print(f"Adding batch {i//batch_size + 1}, documents {i} to {min(i + batch_size, len(docs))}")
        vectorstore.add_documents(batch)
    
    print("Successfully added all documents!")
    
except Exception as e:
    print(f"Error adding documents: {e}")
    # Let's check what the first few documents look like
    print("First document content:")
    print(f"Content length: {len(docs[0].page_content)}")
    print(f"Metadata: {docs[0].metadata}")
    print(f"Content preview: {docs[0].page_content[:200]}...")

llm = OpenAI(temperature=0, openai_api_key=OPENAI_API_KEY)
retriever = MultiQueryRetriever.from_llm(vectorstore.as_retriever(), llm)

Adding batch 1, documents 0 to 5
Adding batch 2, documents 5 to 10
Error adding documents: 5 document(s) failed to index.
First document content:
Content length: 3266
Metadata: {'source': '/Users/malibio/ironhack/lab-chatbot-with-multi-query-retriever/temp.json', 'seq_num': 1, 'name': 'Work From Home Policy', 'summary': 'This policy outlines the guidelines for full-time remote work, including eligibility, equipment and resources, workspace requirements, communication expectations, performance expectations, time tracking and overtime, confidentiality and data security, health and well-being, and policy reviews and updates. Employees are encouraged to direct any questions or concerns', 'url': './sharepoint/Work from home policy.txt', 'category': 'teams', 'updated_at': '2020-03-01'}
Content preview: Effective: March 2020
Purpose

The purpose of this full-time work-from-home policy is to provide guidelines and support for employees to conduct their work remotely, ensuring the continuity an

# Question Answering with MultiQuery Retriever

Now that we have the passages stored in Elasticsearch, we can now ask a question to get the relevant passages.

In [12]:
from langchain.schema.runnable import RunnableParallel, RunnablePassthrough
from langchain.prompts import ChatPromptTemplate, PromptTemplate
from langchain.schema import format_document

import logging

logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

LLM_CONTEXT_PROMPT = ChatPromptTemplate.from_template(
    """You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Be as verbose and educational in your response as possible. 
    
    context: {context}
    Question: "{question}"
    Answer:
    """
)

LLM_DOCUMENT_PROMPT = PromptTemplate.from_template(
    """
---
SOURCE: {name}
{page_content}
---
"""
)


def _combine_documents(
    docs, document_prompt=LLM_DOCUMENT_PROMPT, document_separator="\n\n"
):
    doc_strings = [format_document(doc, document_prompt) for doc in docs]
    return document_separator.join(doc_strings)


_context = RunnableParallel(
    context=retriever | _combine_documents,
    question=RunnablePassthrough(),
)

chain = _context | LLM_CONTEXT_PROMPT | llm

ans = chain.invoke("what is the nasa sales team?")

print("---- Answer ----")
print(ans)

INFO:langchain.retrievers.multi_query:Generated queries: ['1. Can you provide information on the sales team at NASA?', '2. How does the sales team operate within NASA?', '3. What are the responsibilities of the NASA sales team?']


---- Answer ----
I'm sorry, I cannot answer that question as it is not mentioned in any of the provided context. This context is about a company's sales strategy and work from home policies, not about NASA or their sales team.


**Generate at least two new iteratioins of the previous cells - Be creative.** Did you master Multi-
Query Retriever concepts through this lab?

In [13]:
# Creative Iteration 1: Ask about company culture and remote work policies
print("=== QUERY 1: Company Culture & Remote Work ===")

# Ask a question about work-from-home policies
culture_question = "What are the company's work from home policies and how do they support remote employees?"

culture_answer = chain.invoke(culture_question)

print(f"Question: {culture_question}")
print("---- Answer ----")
print(culture_answer)
print("\n" + "="*80 + "\n")

=== QUERY 1: Company Culture & Remote Work ===


INFO:langchain.retrievers.multi_query:Generated queries: ['1. What are the remote work policies of the company and how do they benefit employees working from home?', '2. How does the company support its remote employees through their work from home policies?', "3. Can you provide information on the company's policies for remote work and how they assist employees who work from home?"]


Question: What are the company's work from home policies and how do they support remote employees?
---- Answer ----

The company has a full-time work-from-home policy that was implemented in March 2020 to provide guidelines and support for employees to conduct their work remotely. This policy applies to all eligible employees and is designed to ensure the continuity and productivity of business operations during the COVID-19 pandemic and beyond. 

Under this policy, employees are provided with the necessary equipment and resources for remote work, including a company-issued laptop, software licenses, and access to secure communication tools. Employees are also responsible for creating a comfortable and safe workspace at home that is conducive to productivity. 

Effective communication is also emphasized, and employees are expected to maintain regular communication with their supervisors, colleagues, and team members through various channels. Additionally, employees are expected to main

In [14]:
# Creative Iteration 2: Ask about sales performance and strategy
print("=== QUERY 2: Sales Strategy & Performance ===")

# Ask about sales team performance and strategies
sales_question = "How does the sales team measure success and what strategies do they use to achieve their targets?"

sales_answer = chain.invoke(sales_question)

print(f"Question: {sales_question}")
print("---- Answer ----")
print(sales_answer)
print("\n" + "="*50)

# Bonus: Let's also try a more specific question about team structure
print("=== BONUS QUERY: Team Structure ===")

team_question = "Who are the key people in the sales organization and what are their roles?"

team_answer = chain.invoke(team_question)

print(f"Question: {team_question}")
print("---- Answer ----")
print(team_answer)

=== QUERY 2: Sales Strategy & Performance ===


INFO:langchain.retrievers.multi_query:Generated queries: ['1. What are the key metrics used by the sales team to determine success and what tactics do they employ to reach their goals?', '2. How is success quantified by the sales team and what methods do they utilize to meet their objectives?', '3. What performance indicators are utilized by the sales team to gauge success and what approaches do they implement to attain their desired outcomes?']


Question: How does the sales team measure success and what strategies do they use to achieve their targets?
---- Answer ----

The sales team measures success by establishing key performance indicators (KPIs) to track progress towards their objectives. These KPIs may include metrics such as revenue growth, market share expansion, customer retention rates, and customer satisfaction ratings. They also conduct regular sales team meetings to review performance, share best practices, and address challenges. Additionally, they conduct quarterly reviews of their sales strategy to ensure alignment with market trends and make adjustments as needed.

To achieve their targets, the sales team utilizes various strategies outlined in the sales strategy document. These include expanding the sales team to cover new markets and industries, providing ongoing training to sales staff, implementing a performance-based incentive system, developing targeted marketing campaigns, leveraging digital marketing ch

INFO:langchain.retrievers.multi_query:Generated queries: ['1. Can you provide a list of the main individuals in the sales team and what responsibilities do they hold?', '2. Who are the primary members of the sales organization and what specific functions do they perform?', '3. Could you give me an overview of the key personnel within the sales department and their respective roles?']


Question: Who are the key people in the sales organization and what are their roles?
---- Answer ----

The key people in the sales organization are the sales team, supervisors, and the HR department. The sales team is responsible for executing the sales strategy and achieving the objectives outlined in the document. The supervisors oversee the sales team and provide guidance and support to ensure their success. The HR department plays a crucial role in the sales organization by determining employee eligibility for remote work and providing ongoing training and support to the sales staff. They also collaborate with the sales team to establish performance-based incentives and monitor progress towards the company's objectives.
